In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

In [ ]:
# electric field diagram class
class ElectricField:
    """
    Implementation inspired by 
    https://scipython.com/blog/visualizing-a-vector-field-with-matplotlib/
    """

    # initialise with plot dimensions and resolutions
    def __init__(
        self, cx=0, cy=0, 
        width=200, height=200, 
        x_res=200, y_res=200
    ):

        # plot dimensions
        self.left, self.right = cx - width / 2, cx + width / 2
        self.bottom, self.top = cy - height / 2, cy + height / 2
        
        # plot grid
        x = np.linspace(self.left, self.right, x_res)
        y = np.linspace(self.bottom, self.top, y_res)
        self.X, self.Y = np.meshgrid(x, y)

        # plot formatting
        plt.style.use("dark_background")
        self.fig, self.ax = plt.subplots(figsize=(6,6))
        self.clear_and_build()
        
        # fixed and moving charges
        self.fixeds = []
        self.movers = []

        self.field_lines = None
        self.charge_artists = []

    # clear and rebuild plot
    def clear_and_build(self):

        self.ax.clear()
    
        self.ax.set_xlim(self.left, self.right)
        self.ax.set_ylim(self.bottom, self.top)
        self.ax.set_title(
            "Electric Charges in an Electric Field", 
            fontsize=18
        )
        self.ax.set_xlabel("$x$", fontsize=12)
        self.ax.set_ylabel("$y$", fontsize=12)
    
    # add fixed source charge to plot
    def add_charge(self, q, x, y):
        self.fixeds.append([q, x, y])

    # add free-moving charge to plot
    def add_free_charge(self, q, x, y, m, vx=0, vy=0):
        self.movers.append([q, x, y, m, vx, vy])

    # electric field strength <E_x, E_y>
    def E(self, q, x0, y0, x, y):
        
        dist = np.hypot(x - x0, y - y0)
        factor = q / (4 * np.pi * 8.85e-12 * dist**3)
        
        return factor * (x - x0), factor * (y - y0)

    # Coulomb acceleration on q1 from q2 <F_x, F_y>
    def F(self, q1, q2):

        q1, x1, y1, m, *_ = q1
        q2, x2, y2, *_ = q2

        dist = max(np.hypot(x1 - x2, y1 - y2), 1e-10)
        factor = q1 * q2 / (4 * np.pi * m * 8.85e-12 * dist**3)

        return factor * (x1 - x2), factor * (y1 - y2)

    # electric field vector
    def build_electric_field(self):
        
        Ex = np.zeros_like(self.X)
        Ey = np.zeros_like(self.Y)

        # consider effect of each charge
        for q, x0, y0, *_ in self.fixeds + self.movers:
            ex, ey = self.E(q, x0, y0, self.X, self.Y)
            Ex += ex
            Ey += ey

        return Ex, Ey

    # updated charge positions from Coulomb force
    def move_all(self, dt):

        # find accelerations of charges
        Fx = [0] * len(self.movers)
        Fy = [0] * len(self.movers)
        
        for i, q1 in enumerate(self.movers):
            
            # consider effects of all charges except itself
            for j, q2 in enumerate(self.fixeds + self.movers):
                if i == j - len(self.fixeds): 
                    continue
                fx, fy = self.F(q1, q2)
                Fx[i] += fx
                Fy[i] += fy

        # move all charges
        for i, q in enumerate(self.movers):

            # q = [q, x, y, m, vx, vy]
            q[1] += q[4] * dt + 0.5 * Fx[i] * dt**2
            q[2] += q[5] * dt + 0.5 * Fy[i] * dt**2

            q[4] += Fx[i] * dt
            q[5] += Fy[i] * dt

    # plot electric field
    def plot(self):

        # clear previous frame and rebuild plot
        self.clear_and_build()
        
        Ex, Ey = self.build_electric_field()

        # ensure wide breadth of cmap used
        colour = np.log10(np.hypot(Ex, Ey))
        norm = mcolors.Normalize(
            vmin=np.nanpercentile(colour, 40),
            vmax=np.nanpercentile(colour, 95)
        )

        # plot vector path lines
        self.field_lines = self.ax.streamplot(
            self.X, self.Y, Ex, Ey,
            color=colour,
            linewidth=2,
            cmap=plt.cm.plasma,
            norm=norm,
            density=1.5,
            arrowstyle="->",
            arrowsize=1,
            minlength=0.01
        )

        for q, x0, y0, *_ in self.fixeds + self.movers:

            # colour positive and negative charges different
            colour = "crimson" if q > 0 else "blue"
            dot = self.ax.scatter(
                x0, y0, 
                s=100, 
                color=colour,
                zorder=100
            )

            # annoate with charge
            val = self.ax.annotate(
                f"{q}", (x0, y0), 
                zorder=200, 
                fontsize=8, 
                fontweight="bold", 
                ha="center", 
                va="center"
            )

            self.charge_artists.extend([dot, val])

        return (
            self.field_lines.lines, 
            self.field_lines.arrows, 
            *self.charge_artists
        )

    # animate moving charges
    def animate(self, frames, dt, save=False):

        def update(frame):
            if frame > 0:
                self.move_all(dt)
            return self.plot()

        anim = FuncAnimation(
            self.fig, 
            update, 
            frames=frames, 
            blit=False
        )
        
        if save:
            anim.save("e_field.gif", writer=PillowWriter(fps=10))
        plt.close(self.fig)
        
        # return HTML(anim.to_jshtml())
            
    # save figure as image
    def save(self, title):
        self.fig.savefig(
            f"{title}.png", dpi=300, bbox_inches="tight"
        )